# 05 — Cohort Analysis

Compare user behavior across demographic cohorts:
- Age group × profile completeness
- Sex × bio length
- Education × income
- Drinking behavior × age

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import cohort_summary, ttest_two_groups

In [ ]:
df = pd.read_parquet('../data/processed/okcupid_features.parquet')

## Cohort 1: Age group profiles

In [ ]:
age_cohort = cohort_summary(df, 'age_group', ['bio_length', 'profile_completeness', 'essays_written', 'income'])
age_cohort[['bio_length_mean', 'profile_completeness_mean', 'essays_written_mean', 'income_mean', 'bio_length_count']].round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df.groupby('age_group', observed=True)['bio_length'].median().plot.bar(ax=axes[0], color='#4A90E2')
axes[0].set_title('Median bio length by age group')
axes[0].set_ylabel('Characters')

df.groupby('age_group', observed=True)['profile_completeness'].mean().plot.bar(ax=axes[1], color='#7ED321')
axes[1].set_title('Mean profile completeness by age group')
axes[1].set_ylabel('Fraction')

df.groupby('age_group', observed=True)['income'].median().plot.bar(ax=axes[2], color='#F5A623')
axes[2].set_title('Median income by age group')
axes[2].set_ylabel('USD')

plt.tight_layout()
plt.show()

## Cohort 2: Sex × bio length
Independent two-sample t-test

In [ ]:
result = ttest_two_groups(df, 'sex', 'm', 'f', 'bio_length')
print(f"Mean bio length — male:   {result['mean_a']:.1f} chars (n={result['n_a']:,})")
print(f"Mean bio length — female: {result['mean_b']:.1f} chars (n={result['n_b']:,})")
print(f"t = {result['t_stat']:.3f}, p = {result['p_value']:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x='sex', y='bio_length', data=df[df['bio_length'] < 3000], ax=ax, palette={'m': '#4A90E2', 'f': '#F5A623'})
ax.set_title('Bio length distribution by sex')
ax.set_ylabel('Bio length (characters, clipped at 3000)')
plt.tight_layout()
plt.show()

## Cohort 3: Education × income

In [ ]:
edu_cohort = df.dropna(subset=['education_score', 'income']).groupby('education_score')['income'].agg(['median', 'mean', 'count'])
edu_cohort

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
edu_cohort['median'].plot.bar(ax=ax, color='#9013FE')
ax.set_title('Median income by education level\n(0=working on HS, 5=PhD/Law/Med)')
ax.set_xlabel('Education score')
ax.set_ylabel('Median income (USD)')
plt.tight_layout()
plt.show()

## Cohort 4: Drinking × age

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x='drinks', y='age', data=df,
            order=['not at all', 'rarely', 'socially', 'often', 'very often', 'desperately'],
            ax=ax, color='#4A90E2')
ax.set_title('Age distribution by drinking frequency')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Cross-cohort heatmap: age group × education

In [ ]:
pivot = df.dropna(subset=['education_score']).pivot_table(
    index='age_group', columns='education_score', values='profile_completeness', aggfunc='mean', observed=True,
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax)
ax.set_title('Mean profile completeness by age group × education level')
plt.tight_layout()
plt.show()